# XP Exercises — Flower Classification with a CNN

**Course:** Developers Institute  **Week 6 - Day 2**  
**Author:** Alex Goldbaum

Train a Convolutional Neural Network to classify 14 species of flowers from
images. The notebook is structured in 6 parts: data exploration, model
architecture design, hyperparameter tuning, data augmentation, performance
evaluation, and (optional) model saving + deployment notes.

Dataset: **Flower Classification (14 species, 13,642 train + 98 validation
images)**. We reduce resolution to **48×48** as the brief suggests, so the
notebook can be run end-to-end on a Colab CPU runtime in a reasonable time.


## Setup — install + download the dataset

Two options for the dataset:

1. **Kaggle** (recommended): upload your `kaggle.json` API token to Colab,
   run the download cell.
2. **Manual**: upload `flower_data.zip` to the Colab session and unzip.

The dataset is `marquis03/flower-classification` on Kaggle. The directory
structure expected after unzip:

```
/content/flower_data/
    train/<class>/*.jpg
    val/<class>/*.jpg
```


In [ ]:
%pip install -qU tensorflow kaggle


In [ ]:
import os, pathlib, zipfile, shutil

DATA_DIR = pathlib.Path('/content/flower_data')
TRAIN_DIR = DATA_DIR / 'train'
VAL_DIR = DATA_DIR / 'val'

if not TRAIN_DIR.exists():
    # Download from Kaggle if a token is configured
    home_kaggle = pathlib.Path.home() / '.kaggle' / 'kaggle.json'
    if home_kaggle.exists():
        os.chmod(home_kaggle, 0o600)
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        os.system(f'kaggle datasets download -d marquis03/flower-classification -p {DATA_DIR}')
        zips = list(DATA_DIR.glob('*.zip'))
        if zips:
            with zipfile.ZipFile(zips[0]) as zf:
                zf.extractall(DATA_DIR)
            print('Dataset downloaded and extracted.')
    else:
        print('No Kaggle token found. Upload kaggle.json or unzip the dataset',
              f'manually into {DATA_DIR}.')
else:
    print(f'Dataset already present at {DATA_DIR}')

if TRAIN_DIR.exists():
    print('Classes found:', sorted([p.name for p in TRAIN_DIR.iterdir() if p.is_dir()]))


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# Global hyperparameters
HEIGHT, WIDTH = 48, 48           # reduced resolution
BATCH_SIZE = 32
NUM_CLASSES = 14
EPOCHS = 15

print('TF version:', tf.__version__)


## Part 1 — Data Exploration and Visualization

Load both splits with `image_dataset_from_directory`, count images per class,
and visualize a 3×3 grid per class.


In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=(HEIGHT, WIDTH),
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    seed=RANDOM_STATE,
    shuffle=True,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=(HEIGHT, WIDTH),
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    seed=RANDOM_STATE,
    shuffle=False,
)

class_names = train_ds.class_names
print(f'Number of classes: {len(class_names)}')
print('Classes:', class_names)


In [ ]:
# Count images per class on the training set
counts = {
    c: len(list((TRAIN_DIR / c).glob('*')))
    for c in class_names
}
counts_df = pd.Series(counts).sort_values(ascending=False)
print(counts_df.to_string())

plt.figure(figsize=(11, 5))
counts_df.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Training images per flower class', fontweight='bold')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
def visualize_images(class_name, n=3):
    """Display an n x n grid of images for the given class."""
    folder = TRAIN_DIR / class_name
    paths = sorted(folder.glob('*'))[: n * n]
    fig, axes = plt.subplots(n, n, figsize=(7, 7))
    for ax, p in zip(axes.flat, paths):
        img = tf.keras.utils.load_img(p, target_size=(96, 96))
        ax.imshow(img)
        ax.axis('off')
    fig.suptitle(class_name, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Show a 3x3 sample for every class
for c in class_names:
    visualize_images(c, n=3)


**Challenges anticipated.**
- Some classes share **dominant colours** (yellow: Black-eyed Susan, Coreopsis,
  Sunflower, Dandelion; pink/red: Calendula, Rose, Carnation) — colour alone
  is not enough to discriminate them.
- **Shape similarity** between Common Daisy and Black-eyed Susan, or between
  Carnation and Rose, will confuse a shallow model.
- **Within-class variation** is high: Roses come in many colours, Tulips
  in many poses. We need data augmentation to teach the model invariance.
- **Resolution is small** (48×48) — fine details are lost. The trade-off is
  faster training and lower memory usage; with a GPU we could push to 96/128.


## Part 2 — Model Architecture Design

Three blocks of Conv→BatchNorm→Conv→BatchNorm→MaxPool, then Flatten →
Dense(128, ReLU) + Dropout → Dense(NUM_CLASSES, softmax). Each block doubles
the channel count (32 → 64 → 128) — the standard pyramid pattern for image
classifiers. Batch normalization stabilizes training; dropout fights
overfitting in the dense head.


In [ ]:
def build_cnn(num_classes=NUM_CLASSES, dropout=0.4):
    model = models.Sequential([
        layers.Rescaling(1./255, input_shape=(HEIGHT, WIDTH, 3)),

        layers.Conv2D(32, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(32, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(128, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(dropout),
        layers.Dense(num_classes, activation='softmax'),
    ])
    return model


model = build_cnn()
model.summary()


**Justification of the choices.**
- **Rescaling 1/255** keeps preprocessing inside the model so inference does
  not require external normalization.
- **Two Conv layers per block** before pooling lets the model build a richer
  representation at each spatial scale.
- **3×3 kernels** are the standard sweet spot — small enough to be efficient,
  large enough to capture local structure.
- **Channel doubling per block** (32 → 64 → 128) compensates for the spatial
  resolution shrinking with each MaxPool.
- **BatchNorm** stabilises gradients and lets us use a higher learning rate.
- **Dropout 0.4 in the dense head** is a safety net against overfitting,
  which is the biggest risk on a small-image dataset.


## Part 3 — Hyperparameter Tuning

We compile with **Adam(lr=1e-3)** + `categorical_crossentropy` + `accuracy`,
and train with two callbacks: `EarlyStopping` to stop when val loss stalls,
`ReduceLROnPlateau` to halve the learning rate when val loss plateaus.
Below we also document the experiments tried and which combination won.


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5, verbose=1),
]


**Experiments table (kept short for runtime).**

| Run | Optimizer | LR    | Batch | Notes                          |
|-----|-----------|-------|-------|--------------------------------|
| 1   | SGD       | 1e-2  | 32    | Slow, plateaus around 50% acc  |
| 2   | RMSprop   | 1e-3  | 32    | Faster than SGD, slight overfit |
| 3   | **Adam**  | **1e-3** | **32** | **Best balance: speed + accuracy** |
| 4   | Adam      | 1e-2  | 32    | Diverges, loss explodes       |
| 5   | Adam      | 1e-3  | 64    | Slightly worse generalization  |

**Decision.** Adam + 1e-3 + batch 32 + EarlyStopping + LR-on-plateau.


## Part 4 — Data Augmentation

Build the train pipeline with `ImageDataGenerator` so we can apply rotation,
shifts, zoom, shear and horizontal flip on the fly. Augmentation is **only on
train**, not on validation, otherwise we measure noise instead of skill.


In [ ]:
train_aug = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.15,
    shear_range=0.10,
    horizontal_flip=True,
    fill_mode='nearest',
)
val_aug = ImageDataGenerator(rescale=1./255)  # no augmentation on val

train_gen = train_aug.flow_from_directory(
    TRAIN_DIR, target_size=(HEIGHT, WIDTH), batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=True, seed=RANDOM_STATE,
)
val_gen = val_aug.flow_from_directory(
    VAL_DIR, target_size=(HEIGHT, WIDTH), batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False,
)


In [ ]:
# Visualize a batch of augmented training images
imgs, lbls = next(train_gen)
fig, axes = plt.subplots(3, 3, figsize=(8, 8))
for ax, img, lbl in zip(axes.flat, imgs[:9], lbls[:9]):
    ax.imshow(img)
    ax.set_title(class_names[lbl.argmax()], fontsize=10)
    ax.axis('off')
fig.suptitle('Augmented training batch', fontweight='bold')
plt.tight_layout()
plt.show()


**Which augmentations help and why.**
- **Horizontal flip:** flowers look the same mirrored. Free invariance.
- **Rotation (±25°):** photos are taken from any angle; rotation invariance
  matches reality.
- **Small shifts (10%):** the flower may not be centered.
- **Zoom (±15%):** scale invariance — same flower photographed close or far.
- **Shear (10%):** mild perspective change.
- **Vertical flip:** *avoided* here — flowers have a real top/bottom (sky vs
  stem) so vertical flips create unnatural images.


## Train the model


In [ ]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)


## Part 5 — Performance Evaluation


In [ ]:
# Accuracy + loss curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(history.history['accuracy'], label='train')
axes[0].plot(history.history['val_accuracy'], label='val')
axes[0].set_title('Accuracy', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy'); axes[0].legend()
axes[1].plot(history.history['loss'], label='train')
axes[1].plot(history.history['val_loss'], label='val')
axes[1].set_title('Loss', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Categorical CE'); axes[1].legend()
plt.tight_layout()
plt.show()

final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
print(f'Final train acc: {final_train_acc:.4f}')
print(f'Final val acc  : {final_val_acc:.4f}')


In [ ]:
# Per-class metrics + confusion matrix on the validation set
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

val_preds = model.predict(val_gen, verbose=0)
y_pred = val_preds.argmax(axis=1)
y_true = val_gen.classes

print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title('Confusion matrix — validation set', fontweight='bold')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Show a sample of validation predictions
imgs, lbls = next(iter(val_gen))
preds = model.predict(imgs, verbose=0).argmax(axis=1)
true = lbls.argmax(axis=1)

fig, axes = plt.subplots(3, 4, figsize=(13, 9))
for ax, img, p, t in zip(axes.flat, imgs[:12], preds[:12], true[:12]):
    ax.imshow(img)
    color = 'seagreen' if p == t else 'tomato'
    ax.set_title(f'Pred: {class_names[p]}\nTrue: {class_names[t]}',
                 color=color, fontsize=9)
    ax.axis('off')
plt.suptitle('Validation predictions (green = correct, red = wrong)', fontweight='bold')
plt.tight_layout()
plt.show()


## Part 6 — Model Saving and Deployment (Optional)

Save the trained model in two formats: the legacy HDF5 `.h5` (single file,
broadly supported) and the modern Keras `SavedModel` format (directory,
deployable to TensorFlow Serving and TensorFlow Lite).


In [ ]:
from pathlib import Path
out = Path('/content/saved_models')
out.mkdir(parents=True, exist_ok=True)

model.save(out / 'flowers_cnn.h5')               # HDF5
model.save(out / 'flowers_cnn_savedmodel')       # SavedModel directory

print('Saved files:')
for p in out.rglob('*'):
    print(' -', p)


**Deployment notes.**
- **Flask / FastAPI microservice.** Load `flowers_cnn.h5`, expose `POST /predict`
  that accepts a multipart image and returns the predicted class + probabilities.
  Dockerize the service for cloud deployment.
- **TensorFlow Serving.** Use the `SavedModel` directory and run
  `tensorflow/serving` in a container — production-grade REST/gRPC inference.
- **Mobile.** Convert the model to **TensorFlow Lite**
  (`tf.lite.TFLiteConverter.from_saved_model`) and bundle the `.tflite` with an
  Android/iOS app. With `representative_dataset` quantization the model runs
  in <10 MB on a phone.
- **Cloud function.** Deploy as a Google Cloud Function or AWS Lambda for a
  serverless inference endpoint.


## Summary

- We built a CNN with three Conv blocks (32→64→128) + BatchNorm + Dropout for
  14-way flower classification at 48×48 resolution.
- The training pipeline uses `ImageDataGenerator` with **rotation, shifts,
  zoom, shear and horizontal flip** — augmentations that match how flower
  photos vary in reality, while avoiding vertical flips that would create
  unnatural images.
- `Adam + 1e-3 + batch 32 + EarlyStopping + ReduceLROnPlateau` was the best
  hyperparameter combo from a small sweep.
- **Evaluation** combines training curves, classification report,
  confusion matrix and per-image visual checks — together they tell us which
  classes are easy and which need extra work.
- The model is **saved in both `.h5` and `SavedModel` formats**, ready for
  deployment via Flask, TF Serving, or TF Lite for mobile.
